<a href="https://colab.research.google.com/github/mscampb4-ncsu/Homework7/blob/main/Homework7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7 - ST 554
Author: Max Campbell

## Part 1: Modeling Wine Alcohol Content

In this assignment, we are interested in modeling some of the characteristics present in the [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset provided by the UC Irvine Machine Learning Repository. The data itself is sourced from a sampling of various red and white wines from the north of Portugal. We are focused on two main tasks in the modeling process. The first is trying to regress upon the `alcohol` variable to see how accurately we can predict alcohol content, given some other characteristics of the wine sample. The second is trying to classify a wine as white or red depending on those characteristics. We will start with the former of the two tasks in this part.

As with any dataset, we need to read it in:

In [35]:
#Import necessary packages
import pandas as pd
import numpy as np

#Read in the winequality datasets.
winequalityRed = pd.read_csv("winequality-red.csv", sep = ";")
winequalityRed['color'] = 'red'
winequalityWhite = pd.read_csv("winequality-white.csv", sep = ";")
winequalityWhite['color'] = 'white'

wine = pd.concat([winequalityRed, winequalityWhite], ignore_index = True)
wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,color
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


Note that in the process of reading in the data, the `color` variable was added to represent which dataset the observation came from. This wasn't included initially as the two datasets were seperated by wine color, so there was no need to have a variable in that format that explained the wine color as it would've been redundant.

Now, we can get to the fun part: modeling!

First we will split up the data into a "training set" and a "test set". We fit the models themselves on the training set, which will generally be the majority of the data, and then evaluate the model's performance on data it hasn't seen before in the test set. We will use functionality provided by the `sklearn` module to do this efficiently:

In [46]:
#Import necessary packages for modeling data
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, Ridge, RidgeCV, ElasticNet, ElasticNetCV
from sklearn.preprocessing import PolynomialFeatures, OrdinalEncoder

#Perform a Train/Test split.

X_train, X_test, y_train, y_test = train_test_split(
    wine.drop('alcohol', axis = 1),
    wine['alcohol'],
    test_size = 0.20,
    random_state = 329
)
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,color
2954,7.6,0.19,0.41,1.10,0.040,38.0,143.0,0.99070,2.92,0.42,5,white
5874,5.7,0.22,0.20,16.00,0.044,41.0,113.0,0.99862,3.22,0.46,6,white
2849,5.3,0.26,0.23,5.15,0.034,48.0,160.0,0.99520,3.82,0.51,7,white
371,7.9,0.24,0.40,1.60,0.056,11.0,25.0,0.99670,3.32,0.87,6,red
2762,5.7,0.26,0.27,4.10,0.201,73.5,189.5,0.99420,3.27,0.38,6,white


### Model Training

To begin, let's keep it somewhat simple. We will fit a set of four Multiple Linear Regression (MLR) models of varying complexity.

* Model 1 - A subset of predictors: `fixed acidity`, `citric acid`, `residual sugar`, `pH` and `density`.
* Model 2 - Full predictor model using all 11 numeric predictors shown above.
* Model 3 - Similar to Model 1, but with added interaction terms.
* Model 4 - Similar to Model 1, but with added polynomial terms.

To select the best MLR model, we will use cross-validation to determine which model minimizes the Mean Squared Error.

In [36]:
#MLR Model Cross-validation

cv_mlr_model1 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "citric acid", "residual sugar", "pH", "density"]],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

cv_mlr_model2 = cross_validate(
    LinearRegression(),
    X_train.drop('color', axis = 1),
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

#Create interaction terms to use in Model 3
poly_int = PolynomialFeatures(interaction_only = True, include_bias = False)
design_int =  poly_int.fit_transform(X_train[["fixed acidity", "citric acid", "residual sugar", "pH", "density"]])

cv_mlr_model3 = cross_validate(
    LinearRegression(),
    design_int,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

#Create polynomial terms to use in Model 4
poly = PolynomialFeatures(degree = 2)
design =  poly.fit_transform(X_train[["fixed acidity", "citric acid", "residual sugar", "pH", "density"]])

cv_mlr_model4 = cross_validate(
    LinearRegression(),
    design,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

#Print the results the test scoring on RMSE
print(np.sqrt(-sum(cv_mlr_model1['test_score'])),
      np.sqrt(-sum(cv_mlr_model2['test_score'])),
      np.sqrt(-sum(cv_mlr_model3['test_score'])),
      np.sqrt(-sum(cv_mlr_model4['test_score'])))

1.486839096144431 1.225691746703468 1.4163805947960142 1.2404815241443767


It's close, but it looks like Model 2 was the model that minimized RMSE. As such, we will consider it our 'best' MLR candidate model.

In [37]:
#Fit the MLR candidate model.
mlr_model = LinearRegression().fit(X_train.drop('color', axis = 1), y_train)
print(mlr_model.intercept_)
print(np.array(list(zip(X_train.drop('color', axis = 1).columns, mlr_model.coef_))))

565.7268919655421
[['fixed acidity' '0.5341869842224113']
 ['volatile acidity' '1.5701732755965205']
 ['citric acid' '0.45914892132391316']
 ['residual sugar' '0.1947571555590741']
 ['chlorides' '-0.12831374378088684']
 ['free sulfur dioxide' '-0.0003337720885351336']
 ['total sulfur dioxide' '-0.004059341526990057']
 ['density' '-573.8444890950263']
 ['pH' '2.776199938347132']
 ['sulphates' '1.2797480886601023']
 ['quality' '0.14490306436836123']]


Now, we will apply some more complex models to this set of 11 predictors and evaluate the MLR against these other modeling techniques. note that for all three model types we use, we need to use cross-validation techniques to select a tuning parameter before we can fit the model proper. This helps us account for the complexity of the models by applying the necessary "penalties." First up, a LASSO model.

In [38]:
#Fit a LASSO model
lasso_cv = LassoCV(cv=5, random_state=329) \
    .fit(X_train.drop('color', axis = 1),
         y_train)
lasso_model = Lasso(lasso_cv.alpha_).fit(X_train.drop('color', axis = 1), y_train)
print(lasso_model.intercept_)
print(np.array(list(zip(X_train.drop('color', axis = 1).columns, lasso_model.coef_))))

8.903246299566337
[['fixed acidity' '-0.12424335615641485']
 ['volatile acidity' '-0.0']
 ['citric acid' '0.0']
 ['residual sugar' '-0.06698024707614182']
 ['chlorides' '-0.0']
 ['free sulfur dioxide' '-0.0019275110025548357']
 ['total sulfur dioxide' '-0.0028860417781686604']
 ['density' '-0.0']
 ['pH' '-0.0']
 ['sulphates' '-0.0']
 ['quality' '0.5569036650185127']]


Next up, Ridge Regression!

In [41]:
#Fit a Ridge Regression model
ridge_cv = RidgeCV(cv=5) \
    .fit(X_train.drop('color', axis = 1),
         y_train)
ridge_model = Ridge(ridge_cv.alpha_).fit(X_train.drop('color', axis = 1), y_train)
print(ridge_model.intercept_)
print(np.array(list(zip(X_train.drop('color', axis = 1).columns, ridge_model.coef_))))

54.59227562034463
[['fixed acidity' '-0.10055656468972282']
 ['volatile acidity' '0.9438523784294788']
 ['citric acid' '1.1812701876527132']
 ['residual sugar' '-0.04960965577226117']
 ['chlorides' '-8.618563966398872']
 ['free sulfur dioxide' '-0.0004503500234980884']
 ['total sulfur dioxide' '-0.0046246053453722945']
 ['density' '-46.112813748658056']
 ['pH' '0.06626203576320626']
 ['sulphates' '0.07136444145000102']
 ['quality' '0.4887785820723001']]


And lastly, an Elastic Net model!

In [43]:
#Fit an Elastic Net Model
en_cv = ElasticNetCV(cv=5) \
    .fit(X_train.drop('color', axis = 1),
         y_train)
en_model = ElasticNet(en_cv.alpha_).fit(X_train.drop('color', axis = 1), y_train)
print(en_model.intercept_)
print(np.array(list(zip(X_train.drop('color', axis = 1).columns, en_model.coef_))))

8.969979980623664
[['fixed acidity' '-0.1235257966035188']
 ['volatile acidity' '-0.0']
 ['citric acid' '0.0']
 ['residual sugar' '-0.06698410050242992']
 ['chlorides' '-0.0']
 ['free sulfur dioxide' '-0.0018166529257799397']
 ['total sulfur dioxide' '-0.002913231666689361']
 ['density' '-0.0']
 ['pH' '-0.0']
 ['sulphates' '-0.0']
 ['quality' '0.5445076529132123']]


### Model Testing

Now that we've derived four candidate models, it is time to utilize our testing set! We will use the `predict` function via the `sklearn` module to derive predictions for the `alcohol` content of each observation, and then calculate the RMSE and Mean Absolute Error (MAE) of each set of predictions. The model that minimizes both of these metrics most effectively is our most performant model.

In [49]:
#Predict alcohol content using X_test
mlr_pred = mlr_model.predict(X_test.drop('color', axis = 1))
lasso_pred = lasso_model.predict(X_test.drop('color', axis = 1))
ridge_pred = ridge_model.predict(X_test.drop('color', axis = 1))
en_pred = ridge_model.predict(X_test.drop('color', axis = 1))

#Calculate RMSE for every model
mlr_rmse = np.sqrt(mean_squared_error(y_test, mlr_pred))
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
en_rmse = np.sqrt(mean_squared_error(y_test, en_pred))

#Calculate MAE for every model
mlr_mae = mean_absolute_error(y_test, mlr_pred)
lasso_mae = mean_absolute_error(y_test, lasso_pred)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
en_mae = mean_absolute_error(y_test, en_pred)

#Print results
print("MLR - RMSE: ", mlr_rmse, "--- MAE: ", mlr_mae)
print("LASSO - RMSE: ", lasso_rmse, "--- MAE: ", lasso_mae)
print("Ridge - RMSE: ", ridge_rmse, "--- MAE: ", ridge_mae)
print("Elastic Net - RMSE: ", en_rmse, "--- MAE: ", en_mae)

MLR - RMSE:  0.5028913641858193 --- MAE:  0.38569357184201364
LASSO - RMSE:  0.944707490447724 --- MAE:  0.7545982882906904
Ridge - RMSE:  0.8504043328194646 --- MAE:  0.6817812511065713
Elastic Net - RMSE:  0.8504043328194646 --- MAE:  0.6817812511065713


It appears that keeping the model relatively simple was the way to go. The MLR model clearly performs the best in both RMSE and MAE. As such, it is our best model of the bunch and we can go ahead and fit the full dataset to this model:

In [50]:
#Fit the full dataset to the MLR model:
final_model = LinearRegression().fit(wine.drop(['alcohol', 'color'], axis = 1), wine['alcohol'])
print(final_model.intercept_)
print(np.array(list(zip(wine.drop(['alcohol', 'color'], axis = 1).columns, mlr_model.coef_))))

570.2851913349681
[['fixed acidity' '0.5341869842224113']
 ['volatile acidity' '1.5701732755965205']
 ['citric acid' '0.45914892132391316']
 ['residual sugar' '0.1947571555590741']
 ['chlorides' '-0.12831374378088684']
 ['free sulfur dioxide' '-0.0003337720885351336']
 ['total sulfur dioxide' '-0.004059341526990057']
 ['density' '-573.8444890950263']
 ['pH' '2.776199938347132']
 ['sulphates' '1.2797480886601023']
 ['quality' '0.14490306436836123']]
